In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
import joblib
import os
import warnings
warnings.filterwarnings("ignore")

tickers = [
    "AAPL", "MSFT", "GOOGL", "META", "AMZN", "NFLX", "TSLA", "ABNB",
    "NVDA", "AMD", "INTC", "QCOM", "AVGO", "MU", "ARM", "AMAT",
    "V", "MA", "PYPL", "COIN", "HOOD",
    "JPM", "GS", "BAC", "WFC", "MS", "BLK",
    "CRM", "NOW", "SNOW", "PLTR", "NET", "DDOG", "MDB",
    "WMT", "TGT", "COST", "EBAY", "SHOP",
    "UNH", "ISRG", "DXCM",
    "ENPH", "FSLR", "RIVN"
]
os.makedirs("data/ml", exist_ok=True)
print(f"股票数量: {len(tickers)}")

股票数量: 45


In [2]:
def merge_features_sentiment(ticker):
    feat_path = f"data/features/{ticker}_features.csv"
    sent_path = "data/sentiment/news_with_sentiment.csv"

    if not os.path.exists(feat_path):
        print(f"{ticker}: 特征文件不存在，跳过")
        return None

    df_feat = pd.read_csv(feat_path, index_col=0, parse_dates=True)

    df_sent = pd.read_csv(sent_path)
    df_sent = df_sent[df_sent["ticker"] == ticker].copy()

    if len(df_sent) > 0:
        df_sent["date"] = pd.to_datetime(df_sent["date"], errors="coerce")
        df_sent = df_sent.dropna(subset=["date"])
        daily_sent = df_sent.groupby("date").agg(
            sentiment_mean=("sentiment_numeric", "mean"),
            sentiment_pos_ratio=("sentiment_label", lambda x: (x == "positive").mean()),
            sentiment_neg_ratio=("sentiment_label", lambda x: (x == "negative").mean()),
            news_count=("headline", "count")
        ).reset_index().set_index("date")
        daily_sent.index = pd.to_datetime(daily_sent.index)
        df_merged = df_feat.join(daily_sent, how="left")
    else:
        df_merged = df_feat.copy()
        df_merged["sentiment_mean"] = 0
        df_merged["sentiment_pos_ratio"] = 0
        df_merged["sentiment_neg_ratio"] = 0
        df_merged["news_count"] = 0

    # 填充缺失
    df_merged[["sentiment_mean", "sentiment_pos_ratio",
               "sentiment_neg_ratio", "news_count"]] = \
        df_merged[["sentiment_mean", "sentiment_pos_ratio",
                   "sentiment_neg_ratio", "news_count"]].fillna(0)

    # ② 情感 lag 特征
    df_merged["sentiment_lag1"] = df_merged["sentiment_mean"].shift(1)
    df_merged["sentiment_lag3"] = df_merged["sentiment_mean"].rolling(3).mean()
    df_merged["sentiment_lag5"] = df_merged["sentiment_mean"].rolling(5).mean()

    # ③ news_count log 化
    df_merged["news_count_log"] = np.log1p(df_merged["news_count"])

    df_merged["ticker"] = ticker
    return df_merged

all_dfs = []
for ticker in tickers:
    df = merge_features_sentiment(ticker)
    if df is not None:
        all_dfs.append(df)

df_all = pd.concat(all_dfs)
print(f"合并完成: {len(df_all)} 行, {len(df_all.columns)} 列")
print(f"时间范围: {df_all.index.min()} ~ {df_all.index.max()}")
df_all.head()

合并完成: 77913 行, 29 列
时间范围: 2019-03-14 00:00:00 ~ 2026-07-08 00:00:00


,Open,High,Low,Close,Volume,SMA_20,SMA_50,EMA_20,RSI_14,MACD,...,Target,sentiment_mean,sentiment_pos_ratio,sentiment_neg_ratio,news_count,sentiment_lag1,sentiment_lag3,sentiment_lag5,news_count_log,ticker
Date,,,,,,,,,,,,,,,,,,,,,
2019-03-14,43.821765,43.869423,43.502454,43.781255,94318032,41.657127,39.338955,41.701077,75.107444,0.948176,...,1,0.0,0.0,0.0,0.0,NaN,NaN,NaN,0.0,AAPL
2019-03-15,44.048141,44.639104,43.783638,44.350771,156171648,41.839658,39.476561,41.953429,77.641827,1.044194,...,1,0.0,0.0,0.0,0.0,0.0,NaN,NaN,0.0,AAPL
2019-03-18,44.274518,44.891692,44.272135,44.803525,104879328,42.049354,39.697869,42.224866,79.434430,1.143640,...,0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,AAPL
2019-03-19,44.882161,45.034667,44.303113,44.448471,126585476,42.235221,39.883272,42.436638,74.396876,1.180197,...,1,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,AAPL
2019-03-20,44.376983,45.153813,44.019546,44.836885,124140924,42.427403,40.078008,42.665233,76.176783,1.226373,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,AAPL


In [3]:
feature_cols = [
    # 价格技术指标
    "Close", "Volume",
    "SMA_20", "SMA_50", "EMA_20",
    "RSI_14",
    "MACD", "MACD_signal", "MACD_diff",
    "BB_upper", "BB_lower", "BB_width",
    "Volatility_20",
    # Lag 收益率
    "Return_1d", "Return_3d", "Return_5d",
    # 情感特征（原始）
    "sentiment_mean", "sentiment_pos_ratio", "sentiment_neg_ratio",
    # 情感 lag 特征（新增）
    "sentiment_lag1", "sentiment_lag3", "sentiment_lag5",
    # news count log 化（新增）
    "news_count_log"
]

target_col = "Target"

missing_cols = [c for c in feature_cols if c not in df_all.columns]
if missing_cols:
    print(f"缺失列: {missing_cols}")
else:
    print(f"所有特征列齐全，共 {len(feature_cols)} 个特征")

print(f"\nTarget 分布:\n{df_all[target_col].value_counts(normalize=True).round(3)}")

所有特征列齐全，共 23 个特征

Target 分布:
Target
1    0.522
0    0.478
Name: proportion, dtype: float64


In [4]:
def walk_forward_split(df, n_splits=5):
    df = df.sort_index()
    tscv = TimeSeriesSplit(n_splits=n_splits)
    X = df[feature_cols].values
    dates = df.index
    splits = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
        train_start = dates[train_idx[0]]
        train_end   = dates[train_idx[-1]]
        test_start  = dates[test_idx[0]]
        test_end    = dates[test_idx[-1]]
        splits.append({
            "fold": fold + 1,
            "train_start": train_start, "train_end": train_end,
            "test_start": test_start,   "test_end": test_end,
            "train_size": len(train_idx), "test_size": len(test_idx)
        })
        print(f"Fold {fold+1}: "
              f"Train {train_start.date()}~{train_end.date()} ({len(train_idx)}) | "
              f"Test  {test_start.date()}~{test_end.date()} ({len(test_idx)})")

    return splits

df_nvda = df_all[df_all["ticker"] == "NVDA"][feature_cols + [target_col]].dropna()
print(f"NVDA 样本数: {len(df_nvda)}\n")
splits = walk_forward_split(df_nvda)

NVDA 样本数: 1835

Fold 1: Train 2019-03-20~2020-06-10 (310) | Test  2020-06-11~2021-08-25 (305)
Fold 2: Train 2019-03-20~2021-08-25 (615) | Test  2021-08-26~2022-11-09 (305)
Fold 3: Train 2019-03-20~2022-11-09 (920) | Test  2022-11-10~2024-01-30 (305)
Fold 4: Train 2019-03-20~2024-01-30 (1225) | Test  2024-01-31~2025-04-17 (305)
Fold 5: Train 2019-03-20~2025-04-17 (1530) | Test  2025-04-21~2026-07-08 (305)


In [5]:
df_clean = df_all[feature_cols + [target_col, "ticker"]].dropna()
df_clean = df_clean.sort_index()

print(f"清洗后总样本: {len(df_clean)}")
print(f"覆盖股票:     {df_clean['ticker'].nunique()} 只")

# 手动指定切分点，落在情感数据覆盖窗口（2026-01-28~2026-06-16）中间
# 而不是用全局80%分位数——因为情感数据只覆盖总时间跨度里很小一段，
# 用80%分位数切分会导致所有非零情感数据全部落入测试集，训练集完全学不到情感特征
split_date = pd.Timestamp("2026-03-15")

print(f"情感数据覆盖窗口: 2026-01-28 ~ 2026-06-16")
print(f"选定切分点: {split_date.date()}（位于情感覆盖窗口中间，确保训练/测试集都能见到非零情感样本）")

df_train = df_clean[df_clean.index <  split_date]
df_test  = df_clean[df_clean.index >= split_date]

print(f"\n分割日期:   {split_date.date()}")
print(f"训练集: {len(df_train)} 样本  "
      f"({df_train.index.min().date()} ~ {df_train.index.max().date()})")
print(f"测试集: {len(df_test)}  样本  "
      f"({df_test.index.min().date()} ~ {df_test.index.max().date()})")
print(f"\nTrain Target 分布: {df_train[target_col].value_counts(normalize=True).round(3).to_dict()}")
print(f"Test  Target 分布: {df_test[target_col].value_counts(normalize=True).round(3).to_dict()}")

X_train = df_train[feature_cols]
y_train = df_train[target_col]
X_test  = df_test[feature_cols]
y_test  = df_test[target_col]

清洗后总样本: 77733
覆盖股票:     45 只
情感数据覆盖窗口: 2026-01-28 ~ 2026-06-16
选定切分点: 2026-03-15（位于情感覆盖窗口中间，确保训练/测试集都能见到非零情感样本）

分割日期:   2026-03-15
训练集: 74178 样本  (2019-03-20 ~ 2026-03-13)
测试集: 3555  样本  (2026-03-16 ~ 2026-07-08)

Train Target 分布: {1: 0.522, 0: 0.478}
Test  Target 分布: {1: 0.522, 0: 0.478}


In [6]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=feature_cols,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=feature_cols,
    index=X_test.index
)

print("标准化完成")
print(f"X_train: {X_train_scaled.shape}")
print(f"X_test:  {X_test_scaled.shape}")

标准化完成
X_train: (74178, 23)
X_test:  (3555, 23)


In [7]:
# 特征和标签
X_train_scaled.to_csv("data/ml/X_train.csv")
X_test_scaled.to_csv("data/ml/X_test.csv")
y_train.to_csv("data/ml/y_train.csv")
y_test.to_csv("data/ml/y_test.csv")
df_clean.to_csv("data/ml/full_dataset.csv")

# ④ ticker 元数据（Week 8 SHAP 分析用）
df_train[["ticker"]].to_csv("data/ml/train_meta.csv")
df_test[["ticker"]].to_csv("data/ml/test_meta.csv")

# scaler + schema（QuantTool 推理链路用）
joblib.dump(scaler,            "data/ml/scaler.pkl")
joblib.dump(feature_cols,      "data/ml/feature_cols.pkl")
joblib.dump(feature_cols,      "data/ml/schema.pkl")   # schema 与 feature_cols 一致

print("保存完成，data/ml/ 目录：")
for f in sorted(os.listdir("data/ml")):
    size = os.path.getsize(f"data/ml/{f}") / 1024
    print(f"  {f:30s}  {size:6.1f} KB")

保存完成，data/ml/ 目录：
  X_test.csv                      1643.9 KB
  X_train.csv                     34809.5 KB
  feature_cols.pkl                   0.3 KB
  full_dataset.csv                24315.7 KB
  scaler.pkl                         1.6 KB
  schema.pkl                         0.3 KB
  test_meta.csv                     57.3 KB
  train_meta.csv                  1193.4 KB
  y_test.csv                        48.6 KB
  y_train.csv                     1014.2 KB


In [8]:
print("=== Week 7 ML Preparation Report ===")
print(f"总样本数:          {len(df_clean)}")
print(f"特征数量:          {len(feature_cols)}")
print(f"覆盖股票:          {df_clean['ticker'].nunique()} 只")
print(f"时间范围:          {df_clean.index.min().date()} ~ {df_clean.index.max().date()}")
print(f"训练集:            {len(X_train)} 样本  (截至 {df_train.index.max().date()})")
print(f"测试集:            {len(X_test)}  样本  ({df_test.index.min().date()} 起)")
print(f"分割方式:          按唯一交易日 80/20，无 look-ahead bias")
print(f"情感 lag 特征:     lag1 / rolling3 / rolling5")
print(f"news_count:        log1p 归一化")
print()
print("特征列表:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {col}")
print(df_all["sentiment_mean"].describe())

=== Week 7 ML Preparation Report ===
总样本数:          77733
特征数量:          23
覆盖股票:          45 只
时间范围:          2019-03-20 ~ 2026-07-08
训练集:            74178 样本  (截至 2026-03-13)
测试集:            3555  样本  (2026-03-16 起)
分割方式:          按唯一交易日 80/20，无 look-ahead bias
情感 lag 特征:     lag1 / rolling3 / rolling5
news_count:        log1p 归一化

特征列表:
   1. Close
   2. Volume
   3. SMA_20
   4. SMA_50
   5. EMA_20
   6. RSI_14
   7. MACD
   8. MACD_signal
   9. MACD_diff
  10. BB_upper
  11. BB_lower
  12. BB_width
  13. Volatility_20
  14. Return_1d
  15. Return_3d
  16. Return_5d
  17. sentiment_mean
  18. sentiment_pos_ratio
  19. sentiment_neg_ratio
  20. sentiment_lag1
  21. sentiment_lag3
  22. sentiment_lag5
  23. news_count_log
count    77913.000000
mean         0.000531
std          0.054925
min         -1.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: sentiment_mean, dtype: float64
